# 11 — Validate the susceptibility mart

## Scope

The mart independently selects first qualifying cultures within `base`, ESCE-R, VRE, MRSA, and MSSA categories, with an overall branch only for *S. aureus*. Patients can contribute to several categories; their denominators are not additive.

This notebook checks the category inventory, structural identities, reporting-pair policy, overall *S. aureus* denominator, display rules, and Gram metadata. Obsolete subgroup-within-overall, MRSA+MSSA sum, and all-organism overall-pair checks have been removed, together with the outdated E. coli example. The category inventory now describes the current design instead of testing for an absence of `base` rows.

Unchanged saved outputs remain historical evidence from their original run. Rerun the notebook against the current build for fresh results. Notebook 13 records the saved 2023 result: **424 reporting rows, 194 displayable**. The analysis uses source categorical AST and a project minimum-30 reporting rule; it is not a claim of CLSI certification.


## 1. Inspect the reporting categories and window

Inventory every category without a phenotype filter, grouped by the mart's recorded reporting window. `base` rows are intentional; `overall` is reserved for *S. aureus*. This is a descriptive inventory, not a standalone pass/fail assertion. The updated query has no saved output until rerun.


In [ ]:
%sql
select
    analysis_window_start,
    analysis_window_end_exclusive,
    reporting_phenotype,
    count(distinct organism_reporting_group) as organism_count,
    count(*) as mart_rows,
    sum(case when display_status = 'display' then 1 else 0 end)
        as displayable_rows,
    sum(case when display_status = 'insufficient_data' then 1 else 0 end)
        as insufficient_data_rows
from workspace.micro_dev.mart_susceptibility
group by
    analysis_window_start,
    analysis_window_end_exclusive,
    reporting_phenotype
order by
    analysis_window_start,
    analysis_window_end_exclusive,
    reporting_phenotype;


## 2. Check the reporting-pair seed

Only `reviewed` pairs with `include_primary_dashboard = 1` can reach the mart. Pairs still under review remain in the seed with `review_status = 'proposed'` and are inactive.

In [2]:
%sql
select
    review_status,
    include_primary_dashboard,
    count(*) as pair_rows

from workspace.micro_dev.reportable_organism_antibiotic_pairs

group by
    review_status,
    include_primary_dashboard

order by review_status;

,review_status,include_primary_dashboard,pair_rows
0,proposed,0,11
1,reviewed,1,311


**Reference policy:** the current repository seed contains 311 active/reviewed rows and 11 inactive/proposed rows. The saved query output reflects its execution state; proposals remain inactive until a deliberate reviewed seed change.

## 3. Validate grain and count reconciliation

One mart row represents source release, cohort, organism, reporting category, and antibiotic within the configured window. These structural checks inspect key uniqueness, missing Gram metadata, negative values, and endpoint count identities. The saved 455-row output is historical; current 2023 coverage is 424 rows in Notebook 13. A new execution is needed to claim fresh results from this particular cell.

In [3]:
%sql
select
    count(*) as mart_rows,
    count(distinct susceptibility_mart_key) as unique_keys,
    sum(case when gram_stain_group is null then 1 else 0 end)
        as missing_gram_group,
    sum(case when eligible_count < 0
              or categorized_count < 0
              or untested_count < 0
             then 1 else 0 end) as negative_count_rows,
    sum(case when eligible_count <>
                  categorized_count + inconclusive_count
                  + synergism_count + missing_result_count
                  + conflict_result_count + untested_count
             then 1 else 0 end) as denominator_mismatches,
    sum(case when reporting_phenotype = 'base'
             then 1 else 0 end) as legacy_base_rows
from workspace.micro_dev.mart_susceptibility;

,mart_rows,unique_keys,missing_gram_group,negative_count_rows,denominator_mismatches,legacy_base_rows
0,455,455,0,0,0,0


### Natural-key duplicates

Check for duplicate organism–phenotype–antibiotic combinations within a source release and cohort. Any returned row would require investigation.

In [4]:
%sql
select
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    count(*) as row_count

from workspace.micro_dev.mart_susceptibility

group by
    source_version,
    cohort_id,
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic

having count(*) > 1;

,source_version,cohort_id,organism_reporting_group,organism_name,reporting_phenotype,antibiotic,row_count


### Tested and untested counts

`eligible_count = tested_endpoint_count + untested_count`. This should return no rows.

In [5]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count + untested_count <> eligible_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,eligible_count,tested_endpoint_count,untested_count


### Breakdown of tested endpoints

Reconcile tested endpoints into S/R/I, inconclusive, synergism, missing and conflict; reconcile categorized endpoints into S + I + R. Both comparisons should return no rows.

In [6]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    tested_endpoint_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count

from workspace.micro_dev.mart_susceptibility

where tested_endpoint_count <>
      categorized_count
    + inconclusive_count
    + synergism_count
    + missing_result_count
    + conflict_result_count

   or categorized_count <>
      susceptible_count
    + resistant_count
    + intermediate_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,tested_endpoint_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count


### Component-count bounds

No individual susceptibility category can exceed its denominator. This should return no rows.

In [7]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count

from workspace.micro_dev.mart_susceptibility

where categorized_count > eligible_count
   or susceptible_count > categorized_count
   or resistant_count > categorized_count
   or intermediate_count > categorized_count;

,cohort_id,organism_name,reporting_phenotype,antibiotic,eligible_count,categorized_count,susceptible_count,resistant_count,intermediate_count


## 4. Enforce the reporting-pair policy

The mart should contain only active, reviewed organism–antibiotic pairs in the correct specimen cohort. The exclusion seed takes precedence over the allow-list; suppressed results remain available upstream for analysis.

Match both exclusion scope fields: `*` or the mart row's exact cohort/category. Category-specific rules must not remove an otherwise approved row in another category.


### Inactive or wrong-cohort pairs

Find mart rows without an active matching `(organism_reporting_group, cohort_id, antibiotic)` entry. Expected: no rows.

In [8]:
%sql
select
    m.cohort_id,
    m.organism_reporting_group,
    m.reporting_phenotype,
    m.antibiotic
from workspace.micro_dev.mart_susceptibility m
left join workspace.micro_dev.reportable_organism_antibiotic_pairs p
    on m.organism_reporting_group = p.organism_reporting_group
   and m.cohort_id = p.cohort_id
   and m.antibiotic = p.antibiotic
   and p.include_primary_dashboard = 1
   and p.review_status = 'reviewed'
where p.organism_reporting_group is null;

,cohort_id,organism_reporting_group,reporting_phenotype,antibiotic


### Pairs suppressed by the exclusion seed

Find excluded organism–drug rows within their specified cohort and reporting category. Expected: no rows after the revised seed and mart are rebuilt.


In [ ]:
%sql
select
    m.cohort_id,
    m.organism_reporting_group,
    m.reporting_phenotype,
    m.antibiotic,
    e.exclusion_reason
from workspace.micro_dev.mart_susceptibility m
inner join workspace.micro_dev.organism_antibiotic_exclusions e
    on m.organism_reporting_group = e.organism_reporting_group
   and m.antibiotic = e.antibiotic
   and (e.cohort_id = '*' or m.cohort_id = e.cohort_id)
   and (e.reporting_phenotype = '*'
        or m.reporting_phenotype = e.reporting_phenotype);


## 5. Reconstruct the overall S. aureus denominator

The query counts selected `int_overall_first_reporting_isolate` records for approved, nonsuppressed pairs and compares them with `overall` mart rows. Both sides now concern *S. aureus* only. On a fresh execution against the same build, expect no mismatches. This does not validate base or other independently selected phenotype populations; the current dbt denominator/completeness assertions cover those.

In [ ]:
%sql
with approved_pairs as (
    select
        p.organism_reporting_group,
        p.cohort_id,
        p.antibiotic
    from workspace.micro_dev.reportable_organism_antibiotic_pairs p
    left join workspace.micro_dev.organism_antibiotic_exclusions e
        on p.organism_reporting_group = e.organism_reporting_group
       and p.antibiotic = e.antibiotic
       and (e.cohort_id = '*' or p.cohort_id = e.cohort_id)
       and e.reporting_phenotype in ('*', 'overall')
    where p.review_status = 'reviewed'
      and p.include_primary_dashboard = 1
      and e.organism_reporting_group is null
),
expected as (
    select
        f.source_version,
        f.cohort_id,
        f.organism_reporting_group,
        p.antibiotic,
        count(*) as expected_eligible
    from workspace.micro_dev.int_overall_first_reporting_isolate f
    inner join approved_pairs p
        on f.cohort_id = p.cohort_id
       and f.organism_reporting_group = p.organism_reporting_group
    group by
        f.source_version,
        f.cohort_id,
        f.organism_reporting_group,
        p.antibiotic
),
actual as (
    select
        source_version,
        cohort_id,
        organism_reporting_group,
        antibiotic,
        eligible_count
    from workspace.micro_dev.mart_susceptibility
    where reporting_phenotype = 'overall'
)
select
    coalesce(e.source_version, a.source_version) as source_version,
    coalesce(e.cohort_id, a.cohort_id) as cohort_id,
    coalesce(e.organism_reporting_group, a.organism_reporting_group)
        as organism_reporting_group,
    coalesce(e.antibiotic, a.antibiotic) as antibiotic,
    e.expected_eligible,
    a.eligible_count as actual_eligible
from expected e
full outer join actual a
    on e.source_version = a.source_version
   and e.cohort_id = a.cohort_id
   and e.organism_reporting_group = a.organism_reporting_group
   and e.antibiotic = a.antibiotic
where e.expected_eligible is null
   or a.eligible_count is null
   or e.expected_eligible <> a.eligible_count;


## 6. Validate display rules and Gram metadata

`display_status` is based on the number of categorized S/R/I endpoints. Pairs with fewer than 30 are retained for audit but marked `insufficient_data`; this is a project reporting threshold. Percent susceptible is calculated using categorized results, not all eligible isolates.


In [13]:
%sql
select
    cohort_id,
    organism_name,
    reporting_phenotype,
    antibiotic,
    categorized_count,
    display_status

from workspace.micro_dev.mart_susceptibility

where (
        categorized_count >= 30
        and display_status <> 'display'
      )
   or (
        categorized_count < 30
        and display_status <> 'insufficient_data'
      );

,cohort_id,organism_name,reporting_phenotype,antibiotic,categorized_count,display_status


### Gram-group distribution

Both marts join the reporting-group reference so phenotype categories inherit the base organism's Gram group. The stored 308 Gram-negative and 147 Gram-positive mart rows belong to the earlier full-release snapshot. They are not expected totals for the current 2023 window; use a fresh query for the current distribution.

In [14]:
%sql
select
    gram_stain_group,
    count(distinct organism_reporting_group) as organism_count,
    count(*) as mart_rows

from workspace.micro_dev.mart_susceptibility

group by gram_stain_group

order by gram_stain_group;

,gram_stain_group,organism_count,mart_rows
0,gram_negative,12,308
1,gram_positive,7,147


### Reference-data consistency

Check that every mart row has the expected Gram group according to the reporting-group seed. Expected: no rows.

In [15]:
%sql
select
    m.organism_reporting_group,
    m.organism_name,
    m.gram_stain_group as mart_group,
    g.gram_stain_group as reference_group
from workspace.micro_dev.mart_susceptibility m
left join workspace.micro_dev.organism_reporting_groups g
    on m.organism_reporting_group = g.organism_reporting_group
where m.gram_stain_group is null
   or g.organism_reporting_group is null
   or m.gram_stain_group <> g.gram_stain_group;

,organism_reporting_group,organism_name,mart_group,reference_group


## 7. Inspect the reporting shape

### Blood-cohort preview

Review eligible and categorized denominators alongside the AST counts. The saved preview belongs to its earlier execution. The current display policy retains insufficient-data rows for audit while suppressing or clearly marking their percentages in Tableau; a blank or insufficient-data cell must not become 0% susceptibility.


In [16]:
%sql
select
    organism_reporting_group,
    organism_name,
    reporting_phenotype,
    antibiotic,
    eligible_count,
    tested_endpoint_count,
    untested_count,
    categorized_count,
    susceptible_count,
    resistant_count,
    intermediate_count,
    inconclusive_count,
    synergism_count,
    missing_result_count,
    conflict_result_count,
    categorized_result_coverage_pct,
    susceptible_pct,
    resistant_pct,
    display_status

from workspace.micro_dev.mart_susceptibility

where cohort_id = 'blood'

order by
    organism_reporting_group,
    reporting_phenotype,
    pair_order;

,organism_reporting_group,organism_name,reporting_phenotype,antibiotic,eligible_count,tested_endpoint_count,untested_count,categorized_count,susceptible_count,resistant_count,intermediate_count,inconclusive_count,synergism_count,missing_result_count,conflict_result_count,categorized_result_coverage_pct,susceptible_pct,resistant_pct,display_status
0,Citrobacter freundii complex,Citrobacter freundii complex,overall,Ciprofloxacin,84,84,0,84,78,5,1,0,0,0,0,100.0,92.9,6.0,display
1,Citrobacter freundii complex,Citrobacter freundii complex,overall,Gentamicin,84,84,0,84,83,1,0,0,0,0,0,100.0,98.8,1.2,display
2,Citrobacter freundii complex,Citrobacter freundii complex,overall,Trimethoprim/Sulfamethoxazole,84,84,0,84,75,9,0,0,0,0,0,100.0,89.3,10.7,display
3,Citrobacter freundii complex,Citrobacter freundii complex,overall,Cefepime,84,50,34,50,49,1,0,0,0,0,0,59.5,98.0,2.0,display
4,Citrobacter freundii complex,Citrobacter freundii complex,overall,Meropenem,84,83,1,83,83,0,0,0,0,0,0,98.8,100.0,0.0,display
5,Citrobacter freundii complex,Citrobacter freundii complex,overall,Ertapenem,84,59,25,59,59,0,0,0,0,0,0,70.2,100.0,0.0,display
6,Citrobacter freundii complex,Citrobacter freundii complex,overall,Levofloxacin,84,81,3,81,76,3,2,0,0,0,0,96.4,93.8,3.7,display
7,Citrobacter koseri,Citrobacter koseri,overall,Ceftriaxone,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display
8,Citrobacter koseri,Citrobacter koseri,overall,Ciprofloxacin,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display
9,Citrobacter koseri,Citrobacter koseri,overall,Gentamicin,44,44,0,44,44,0,0,0,0,0,0,100.0,100.0,0.0,display


## 8. Interpretation and current validation record

Structural and policy checks should pass against a consistent build. Each reporting category has its own selected population: non-S. aureus categories do not require an `overall` row, and MRSA plus MSSA counts may exceed overall *S. aureus* because the same patient can qualify for both.

Use `assert_reporting_mart_population_complete` for the category-specific approved-pair grid and denominator counts, and `assert_stratified_denominators_within_population` to bound each category by the eligible patient–organism universe. Exclusions now match organism, antibiotic, cohort, and reporting category. A `*` scope matches all values of that dimension; count the surviving category-specific grid rather than subtracting the number of rules from the approval count.

Notebook 13 records **424 rows: 194 displayable and 230 insufficient-data rows** for the 2023 shifted-date window. The saved full dbt build records **22 models, 4 seeds, and 224 passing tests**. These are recorded results; the cleanup did not run a new warehouse build or refresh the retained historical outputs. Notebook 12 checks the separate MDRO population.


The 424-row result predates the 28 scoped exclusions approved on October 13, 2026. Applying them to that saved export removes exactly 28 rows (7 displayable and 21 insufficient-data), projecting 396 rows, 187 displayable and 209 insufficient-data. This is an offline projection until the revised seed and mart are rebuilt and validated.
